# STT 07 — Entraînement sur le grand dataset (*Scripted Speech 27.0* FR, 894 h)

**Ce qui change par rapport à l'essai 1 :** seulement les **données**. On passe du petit dataset (19 h, 275 voix) au **train officiel du grand dataset** (894 h, 10 295 voix), et on évalue sur le **dev officiel** (15 944 clips, 3 275 voix). Le modèle, l'optimizer, SpecAugment et le scheduler sont les mêmes.

| Réglage | Valeur | D'où ça vient |
|---|---|---|
| Modèle | S (3,6 M de paramètres), dropout 0,1 | Inchangé |
| Batch | 32 | Inchangé |
| Optimizer | AdamW, lr maximum 1e‑3, weight decay 0,01 | Inchangé |
| Scheduler | warmup **5 %** (environ 19 100 steps), puis cosinus jusqu'à 0 | Décision du 10/11 |
| Durée | **20 époques**, environ 382 700 steps, environ 21 h | Décision du 10/11 (option 3), confirmée par la mesure de `stt_06` |
| Clipping | 1,0 | Inchangé |
| Augmentation | SpecAugment sur le train | Essai 1 |
| Filtre | 113 clips du train retirés (contrainte du CTC) | EDA du grand dataset, point 5 |
| Sauvegardes | `last.pt` tous les 2 000 steps, **arrêt automatique à 11 h 15** de session | Décision du 10/11 (option B) |
| Évaluation | dev officiel **toutes les demi-époques** ; `best.pt` = meilleur WER | Décision du 10/11 |

**Deux sessions de Kaggle :** une session dure 12 h au maximum. La session 1 fait environ 10 époques, puis s'arrête seule. La session 2 **reprend exactement là où la 1re s'est arrêtée**, au milieu d'une époque s'il le faut.

**Avant de lancer (panneau de droite) :**
1. **Accelerator → GPU T4 x2** (notre code utilise un seul des deux GPU). **Internet → On.** **Secrets** : coche le Secret de la clé MDC. Notebook en **Private**.
2. **Session 1 :** Save Version → **Save & Run All**.
3. **Session 2 :** après la fin de la session 1, ouvre le notebook en édition, puis **Add Input** (panneau de droite) → onglet *Your Work* → choisis **ce notebook** (la version de la session 1). Ses fichiers `last.pt`, `best.pt` et `history.json` apparaissent dans `/kaggle/input/`. Tu peux laisser les inputs précédents (par exemple un checkpoint venu de Colab) : s'il y a plusieurs `last.pt`, la cellule 4 prend **le plus avancé** (le plus grand nombre de steps). Relance ensuite **Save & Run All**.

**Règles de données (conditions MDC) :** les données vont dans `/tmp`, jamais sauvegardé avec le notebook. Seuls les **poids du modèle** et les **chiffres** sont sauvegardés dans *Output*. Aucune phrase du dataset n'est affichée.

**À renvoyer après chaque session :** les lignes du log qui commencent par `ép`, ou le fichier `history.json`.

## Cellule 1 — Notre code, la machine, et le chronomètre de la session

`SESSION_START` est lancé **ici**, tout au début : la limite de Kaggle (12 h) compte depuis le démarrage de la session, téléchargement compris. C'est par rapport à lui que la boucle décidera de s'arrêter à 11 h 15.

In [ ]:
import time
SESSION_START = time.time()            # le chronomètre de la session (limite Kaggle : 12 h)

!git clone -b stt-transformer-junior --depth 1 https://github.com/mohamed-943/projet-J-ONE.git
%cd projet-J-ONE/recherche/stt_transformer
!pip install -q datacollective soundfile
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Cellule 2 — Téléchargement et décompression (environ 22 min)

Comme dans `stt_05` et `stt_06`. Ce temps se répète au début de **chaque** session : les conditions MDC interdisent d'enregistrer les données en dataset Kaggle.

In [ ]:
import os, glob, json, shutil, subprocess
from kaggle_secrets import UserSecretsClient

# La clé est lue dans le Secret Kaggle : elle n'apparaît jamais dans le code.
# Kaggle cherche le nom EXACT du Secret : on accepte les deux écritures.
secrets = UserSecretsClient()
try:
    os.environ["MDC_API_KEY"] = secrets.get_secret("MDC_API_KEY")
except Exception:
    os.environ["MDC_API_KEY"] = secrets.get_secret("mdc_api_key")
from datacollective import download_dataset

BIG_ID = "cmu5kub4600pkmh078n7n5ogl"           # Scripted Speech 27.0 FR
BIG_DIR = "/tmp/big"                           # /tmp : jamais sauvegardé avec le notebook (conditions MDC)

download_dataset(BIG_ID, download_directory=BIG_DIR)
archive = glob.glob(f"{BIG_DIR}/*.tar.gz")[0]
subprocess.run(["tar", "-xzf", archive, "-C", BIG_DIR], check=True)
os.remove(archive)                             # libère les 30 Go de l'archive
BASE = os.path.dirname(glob.glob(f"{BIG_DIR}/**/train.tsv", recursive=True)[0])
print(f"Données prêtes après {(time.time() - SESSION_START) / 60:.1f} min de session")

## Cellule 3 — Les données : filtre du CTC, ordre reproductible, DataLoaders

**1. Le filtre du CTC (EDA, point 5).** Le CTC exige au moins un instant de sortie par caractère, plus un blank entre deux lettres identiques. Après la réduction ÷ 4, le modèle dispose de 25 instants par seconde. On retire donc du **train** :
- les clips qui demandent **plus de 25** instants par seconde : impossibles, ce sont des transcriptions fausses ;
- ceux qui en demandent **moins de 2** : un audio bien trop long pour son texte, suspect.

C'est exactement la règle de l'EDA, qui en avait compté 113. Le **dev** n'est pas filtré : c'est le jeu officiel, on le garde intact pour que le WER reste comparable.

**2. Un ordre aléatoire, mais reproductible.** Pour reprendre **au milieu d'une époque** (session 2), il faut savoir quels clips ont déjà été vus. Un simple `shuffle=True` tire un nouvel ordre à chaque fois, impossible à retrouver. Notre `ResumableSampler` tire l'ordre de l'époque *e* avec une graine fixe (`seed + e`) : la même époque donne **toujours le même ordre**. Pour reprendre au batch *b*, il saute simplement les *b* premiers paquets de 32 clips, sans les lire.

*Vocabulaire :* un **sampler** est l'objet qui dit au DataLoader **dans quel ordre** lire les clips. Le DataLoader, lui, les lit et les met en batch.

In [ ]:
import math
import pandas as pd
import torch
from torch.utils.data import DataLoader, Sampler
from stt.dataset import load_split, CommonVoiceDataset, collate_batch

BATCH_SIZE = 32
train_table = load_split([BASE], "train")      # lecture du .tsv, normalisation, retrait des phrases hors vocabulaire
dev_table = load_split([BASE], "dev")

# --- 1. Filtre du CTC (même règle que l'EDA, point 5) ---
durations = {}
if os.path.exists(f"{BASE}/clip_durations.tsv"):              # présent dans le grand dataset (vérifié par l'EDA)
    d = pd.read_csv(f"{BASE}/clip_durations.tsv", sep="\t")
    durations = dict(zip(d.iloc[:, 0], d.iloc[:, 1] / 1000))   # nom du mp3 -> secondes (fichier en millisecondes)
else:
    print("ATTENTION : clip_durations.tsv absent, le filtre du CTC est désactivé")

def ctc_needed(text):
    # instants minimum pour le CTC : un par caractère, plus un blank entre deux lettres identiques
    return len(text) + sum(a == b for a, b in zip(text, text[1:]))

seconds = train_table["audio_path"].map(lambda p: durations.get(os.path.basename(p)))
rate = train_table["text"].map(ctc_needed) / seconds           # instants nécessaires par seconde
keep = rate.isna() | ((rate >= 2) & (rate <= 25))              # durée inconnue : on garde
print(f"Filtre du CTC : {(~keep).sum()} clips retirés (l'EDA en annonçait 113)")
train_table = train_table[keep].reset_index(drop=True)

# --- 2. Un ordre aléatoire reproductible, qui peut repartir du milieu d'une époque ---
class ResumableSampler(Sampler):
    def __init__(self, n_clips, batch_size, seed=0):
        self.n_clips, self.batch_size, self.seed = n_clips, batch_size, seed
        self.epoch, self.start_batch = 0, 0

    def set_epoch(self, epoch, start_batch=0):
        # appelé au début de chaque époque ; start_batch > 0 seulement lors d'une reprise
        self.epoch, self.start_batch = epoch, start_batch

    def __iter__(self):
        g = torch.Generator().manual_seed(self.seed + self.epoch)      # même époque -> même graine -> même ordre
        order = torch.randperm(self.n_clips, generator=g).tolist()     # une permutation des indices des clips
        return iter(order[self.start_batch * self.batch_size:])        # on saute les batchs déjà faits

    def __len__(self):
        return self.n_clips - self.start_batch * self.batch_size

train_sampler = ResumableSampler(len(train_table), BATCH_SIZE)
train_loader = DataLoader(CommonVoiceDataset(train_table, augment=True), batch_size=BATCH_SIZE,
                          sampler=train_sampler, collate_fn=collate_batch, num_workers=4)
dev_loader = DataLoader(CommonVoiceDataset(dev_table, augment=False), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_batch, num_workers=4)

STEPS_PER_EPOCH = math.ceil(len(train_table) / BATCH_SIZE)
print(f"Train : {len(train_table):,} clips -> {STEPS_PER_EPOCH:,} steps par époque | dev : {len(dev_table):,} clips")

## Cellule 4 — Modèle, optimizer, scheduler, et reprise depuis `last.pt`

**Les réglages** sont regroupés en haut de la cellule, en majuscules.

**Ce que contient `last.pt`** (tout ce qu'il faut pour reprendre exactement) :
- les **poids** du modèle ;
- l'état de l'**optimizer** : AdamW garde, pour chaque poids, une moyenne des gradients passés et de leurs carrés. Sans eux, la reprise repartirait « à froid », avec de gros pas désordonnés ;
- l'état du **scheduler** : le step où il en est, donc le bon learning rate ;
- notre **état** : le step global, l'époque, le batch dans l'époque, le meilleur WER et l'historique des mesures.

**`best.pt`** contient seulement les poids du modèle qui a le meilleur WER sur le dev. C'est lui qu'on utilisera ensuite.

**La reprise :** la cellule cherche `last.pt` d'abord dans `/kaggle/working` (la session en cours), puis dans `/kaggle/input` (la session précédente, ajoutée avec *Add Input*). S'il n'y en a pas, l'entraînement commence de zéro.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR
from stt.model import SpeechTransformer
from stt.training import warmup_cosine, evaluate

# --- Réglages (décisions du journal, 10/11) ---
EPOCHS = 20
TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = int(0.05 * TOTAL_STEPS)         # warmup de 5 %
SAVE_EVERY = 2000                              # last.pt tous les 2 000 steps (environ 6 min)
EVAL_AT = {STEPS_PER_EPOCH // 2, STEPS_PER_EPOCH}   # évaluation au milieu et à la fin de chaque époque
TIME_BUDGET_H = 11.25                          # arrêt automatique à 11 h 15 de session (limite Kaggle : 12 h)
OUT = "/kaggle/working"                        # les fichiers d'ici sont sauvegardés dans Output

device = "cuda"
torch.manual_seed(0)
model = SpeechTransformer().to(device)                       # taille S
ctc_loss = torch.nn.CTCLoss(blank=0, zero_infinity=True)     # blank = indice 0 du vocabulaire
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)   # weight decay 0,01 par défaut
scheduler = LambdaLR(optimizer, lambda step: warmup_cosine(step, WARMUP_STEPS, TOTAL_STEPS))

state = {"step": 0, "epoch": 0, "batch": 0, "best_wer": float("inf"), "history": []}

def save_checkpoint():
    # On écrit d'abord un fichier temporaire, puis on le renomme : si la session est coupée
    # pendant l'écriture, l'ancien last.pt reste intact (jamais de fichier à moitié écrit).
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "state": state, "total_steps": TOTAL_STEPS},
               f"{OUT}/last.tmp")
    os.replace(f"{OUT}/last.tmp", f"{OUT}/last.pt")

# --- Reprise ---
candidates = [f"{OUT}/last.pt"] + sorted(glob.glob("/kaggle/input/**/last.pt", recursive=True))
# S'il y a plusieurs last.pt (dataset de départ + sorties de sessions précédentes), on prend le plus avancé
checkpoints = [(torch.load(p, map_location=device, weights_only=False), p) for p in candidates if os.path.exists(p)]
found = None
if checkpoints:
    ckpt, found = max(checkpoints, key=lambda c: c[0]["state"]["step"])
    print("last.pt trouvés :", [(p, c["state"]["step"]) for c, p in checkpoints])
    del checkpoints                                   # libère la mémoire des autres checkpoints
if found:
    assert ckpt["total_steps"] == TOTAL_STEPS, "Le nombre total de steps a changé : le scheduler ne serait plus le même"
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    state = ckpt["state"]
    # On recopie best.pt et history.json de la session précédente, pour les retrouver dans le nouvel Output
    for name in ("best.pt", "history.json"):
        src = os.path.join(os.path.dirname(found), name)
        if os.path.dirname(found) != OUT and os.path.exists(src):
            shutil.copy(src, f"{OUT}/{name}")
    print(f"Reprise depuis {found} : step {state['step']:,}, époque {state['epoch'] + 1}, "
          f"batch {state['batch']:,}/{STEPS_PER_EPOCH:,}, meilleur WER {state['best_wer']:.1f}%")
else:
    print("Aucun last.pt : l'entraînement commence de zéro")
print(f"Steps prévus : {TOTAL_STEPS:,} dont {WARMUP_STEPS:,} de warmup")

## Cellule 5 — La boucle d'entraînement

Pour chaque batch, les 6 étapes habituelles : **forward → loss CTC → zero_grad → backward → clipping → optimizer + scheduler**.

Autour de ces étapes, trois rendez-vous :
- **toutes les demi-époques** : évaluation sur le dev officiel (environ 1,2 min). On enregistre une ligne dans l'historique (`history.json`) et on sauvegarde `best.pt` si le WER est le meilleur jusqu'ici (★) ;
- **tous les 2 000 steps** : sauvegarde de `last.pt` ;
- **après chaque step** : si la session a dépassé 11 h 15, on sauvegarde `last.pt` et on s'arrête proprement. La session 2 reprendra à partir de là.

**Ce qu'affiche le log :**
- tous les 1 000 steps, une ligne de progression ;
- à chaque évaluation, une ligne qui commence par `ép`. La loss train y est la moyenne depuis l'évaluation précédente (sur des spectrogrammes **masqués** par SpecAugment, donc plus difficiles que le dev).

Aucune phrase du dataset n'est affichée : le log peut être recopié sans risque.

In [ ]:
def time_is_up():
    return time.time() - SESSION_START > TIME_BUDGET_H * 3600

model.train()
stopped_early = False
for epoch in range(state["epoch"], EPOCHS):
    train_sampler.set_epoch(epoch, state["batch"])          # même ordre qu'avant la coupure, en sautant les batchs faits
    loss_sum, grad_norms, t_start = 0.0, [], time.time()

    for batch in train_loader:
        features, feature_lengths, targets, target_lengths = [t.to(device) for t in batch]
        log_probs, out_lengths = model(features, feature_lengths)                        # forward
        loss = ctc_loss(log_probs.transpose(0, 1), targets, out_lengths, target_lengths)  # (T, B, C) pour le CTC
        optimizer.zero_grad()
        loss.backward()                                                                   # backpropagation
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)               # norme avant clipping
        optimizer.step()
        scheduler.step()

        state["step"] += 1
        state["batch"] += 1
        loss_sum += loss.item()
        grad_norms.append(grad_norm.item())

        if state["step"] % 1000 == 0:
            print(f"  step {state['step']:,}/{TOTAL_STEPS:,} | loss {loss_sum / len(grad_norms):.3f} | "
                  f"lr {scheduler.get_last_lr()[0]:.1e} | session {(time.time() - SESSION_START) / 3600:.2f} h")

        if state["batch"] in EVAL_AT:                         # milieu ou fin de l'époque
            dev = evaluate(model, dev_loader, ctc_loss, device)
            record = {"epoch": round(epoch + state["batch"] / STEPS_PER_EPOCH, 2), "step": state["step"],
                      "train_loss": round(loss_sum / len(grad_norms), 4), "dev_loss": round(dev["loss"], 4),
                      "wer": round(dev["wer"], 4), "cer": round(dev["cer"], 4),
                      "lr": scheduler.get_last_lr()[0],
                      "grad_mean": round(sum(grad_norms) / len(grad_norms), 3), "grad_max": round(max(grad_norms), 2),
                      "minutes": round((time.time() - t_start) / 60, 1)}
            state["history"].append(record)
            mark = ""
            if dev["wer"] < state["best_wer"]:
                state["best_wer"] = dev["wer"]
                torch.save({"model": model.state_dict(), "step": state["step"], "wer": dev["wer"]}, f"{OUT}/best.pt")
                mark = "  ★ meilleur"
            with open(f"{OUT}/history.json", "w") as f:
                json.dump(state["history"], f, indent=1)
            print(f"ép {record['epoch']:5.2f} | train {record['train_loss']:.3f} | dev {record['dev_loss']:.3f} | "
                  f"WER {record['wer']:5.1f}% | CER {record['cer']:5.1f}% | lr {record['lr']:.1e} | "
                  f"grad moy {record['grad_mean']:.2f} max {record['grad_max']:.1f} | {record['minutes']} min{mark}")
            loss_sum, grad_norms, t_start = 0.0, [], time.time()

        if state["step"] % SAVE_EVERY == 0:
            save_checkpoint()
        if time_is_up():
            save_checkpoint()
            stopped_early = True
            break

    if stopped_early:
        break
    state["epoch"], state["batch"] = epoch + 1, 0             # époque terminée : la suivante commence au batch 0
    save_checkpoint()

## Cellule 6 — Bilan de la session

On indique si l'entraînement est **terminé**, ou s'il faut lancer **la session suivante**.

In [ ]:
print(f"Session : {(time.time() - SESSION_START) / 3600:.2f} h | step {state['step']:,}/{TOTAL_STEPS:,} "
      f"| meilleur WER dev : {state['best_wer']:.1f}%")
if state["step"] < TOTAL_STEPS:
    print("Entraînement NON terminé : lance la session suivante (Add Input -> cette version, puis Save & Run All).")
else:
    print("Entraînement terminé : best.pt (meilleur WER dev) et history.json sont dans Output.")